In [1]:
import sys
sys.path.append("/project/src")

In [ ]:
# Helper imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math

from sklearn.model_selection import train_test_split
from sksurv.util import Surv

from rsfmodel import RandomSurvivalForest
from preprocessing import (
    BaseDatasetPreprocessor,
    build_preprocessing_pipeline,
    build_survival_target,
    clean_columns,
    create_missingness_indicators,
    decode_preprocessed_feature_name,
    define_categorical_and_continuous_columns,
    define_missingness,
    drop_useless_columns,
    filter_columns_by_missing_pattern,
    keep_available,
    low_missingness_complete_case_analysis,
    select_features_subset,
    split_features_target,
    IMPORTANT_VERY_IMBALANCED_COLUMNS,
    LOW_MISSINGNESS_THRESHOLD,
    HIGH_MISSINGNESS_THRESHOLD,
    MANDATORY_FEATURE,
    N_IMPORTANT_FEATURES,
    NOT_COLLECTED_PLACEHOLDER_VALUE,
    SURVIVAL_EVENT_COL,
    SURVIVAL_TIME_COL,
)

In [ ]:
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive # type: ignore
    drive.mount('/content/drive')
    nacc_data_csv = "/content/drive/MyDrive/bachelor/nacc_data_2025.csv"
else:
    nacc_data_csv = "./data/nacc_data_2025.csv"

In [4]:
nacc_raw = pd.read_csv(nacc_data_csv, delimiter=',')

/tmp/ipykernel_29/2070981678.py:1: DtypeWarning: Columns (8,10,12,14,25,29,31,40,42,44,46,48,50,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,142,194,197,199,205,207,209,211,213,215,217,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,374,376,378,398,400,409,422,429,469,549,572,580,605,656,673,676,693,710,763,774,776,778,780,786,847,902,903,904,912,913,914,915,925,947,950,953) have mixed types. Specify dtype option on import or set low_memory=False.
  nacc_raw = pd.read_csv(nacc_data_csv, delimiter=',')


In [6]:
nacc_raw.head()

,NACCID,VISITMO,VISITDAY,VISITYR,NACCREFR,SEX,HISPANIC,HISPOR,HISPORX,RACE,...,NPATGAM3,NPATGAM4,NPATGAM5,NPATGFRN,NPATGFR1,NPATGFR2,NPATGFR3,NPATGFR4,EVENT_MCI,TIME
0,NACC003487,11,15,2023,2.0,1,0.0,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
1,NACC004687,11,14,2022,2.0,1,1.0,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
2,NACC013204,8,7,2023,2.0,1,1.0,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1
3,NACC014621,2,22,2024,2.0,2,1.0,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
4,NACC014791,12,12,2023,2.0,2,1.0,1.0,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,1


In [7]:
print(f"shape {nacc_raw.shape}")

shape (29673, 969)


According to EPV rule we need to provide as least 10 records per variable for the analysis and prediction to be reliable. After analysing the number of events during the eda - the number of features cannot extend 237 values (The 1 place is remained for the missing indicator, that should definetelly be in the final dataset)

## Feature selection

In [ ]:
def rsf_vimp_feature_selection(x_data, y_data, n_features, n_estimators=15):
    rsf = RandomSurvivalForest(
        num_trees       = n_estimators,
        min_node_size   = 20,
        mtry            = int(np.sqrt(x_data.shape[1])),
        importance      = "permutation",
        compute_weights = True
    )
    rsf.fit(x_data, y_data)
    vimp_importance = rsf.get_importance()
    return vimp_importance.head(n_features).index.tolist()


# Dataset preprocessor class

In [ ]:
class FullDatasetPreprocessor(BaseDatasetPreprocessor):
    def __init__(self, n_important_features=N_IMPORTANT_FEATURES, best_rsf_vimp_parameters=None):
        super().__init__()
        self.n_important_features     = n_important_features
        self._selected_features       = None
        self._train_categorical_cols  = None
        self._train_continuous_cols   = None
        self.best_parameters          = best_rsf_vimp_parameters
  
    def fit_transform(self, df):
        """
          Fit the full pipeline on df and return (X, y).
          Extends the base fit with feature selection and subset re-fitting.

          Steps
          -----
          1–3. Base preprocessing (see :class:`BaseDatasetPreprocessor`).
          4.   RSF-VIMP feature selection → select top-N features.
          5.   Complete-case analysis restricted to the selected feature subset.
          6.   Re-fit + apply the pipeline on the reduced subset.
        """
        X_preprocessed, y_full = super().fit_transform(df)

        # Snapshot column roles before they are overwritten by subset re-fit
        self._train_categorical_cols  = list(self._categorical_cols)
        self._train_continuous_cols   = list(self._continuous_cols)
        self._selected_features       = self._select_features(X_preprocessed, y_full)
        self._fitted_columns          = list(X_preprocessed.columns)

        X_final, y_final = self._build_subset_pipeline(
            selected_features=self._selected_features,
            categorical_cols=self._train_categorical_cols,
            continuous_cols=self._train_continuous_cols,
            pipeline_mode="fit_transform",
        )
        return X_final, y_final

    def transform(self, df):
        """
          Apply the already-fitted full pipeline to df and return (X, y).
        """
        self._assert_fully_fitted()
        self._apply_structural_cleanup(df)

        X_final, y_final = self._build_subset_pipeline(
            selected_features=self._selected_features,
            categorical_cols=self._train_categorical_cols,
            continuous_cols=self._train_continuous_cols,
            pipeline_mode="transform",
        )
        return X_final, y_final

    def _select_features(self, X, y):
        """
          Select subset of features to reduce the dimensionality of the dataset
          Applicable only on training data.
        """
        selected = rsf_vimp_feature_selection(
            X, y, n_features=self.n_important_features, parameters=self.best_parameters
        )
        if MANDATORY_FEATURE not in selected:
            selected.append(MANDATORY_FEATURE)
        return selected

    def _build_subset_pipeline(self, selected_features, categorical_cols, continuous_cols, pipeline_mode):
        """
          Reverse-map selected preprocessed feature names to raw columns, apply
          complete-case analysis on the subset, and re-run the pipeline.
        """
        raw_subset = select_features_subset(
            self._cleaned_df,
            selected_features,
            categorical_cols,
            continuous_cols,
        )

        subset_low_missing = keep_available(self._low_missing_cols, raw_subset.columns)
        subset_cleaned     = low_missingness_complete_case_analysis(raw_subset, subset_low_missing)
        X_subset, y_subset = split_features_target(subset_cleaned)

        if pipeline_mode == "fit_transform":
            X_preprocessed = self._fit_transform_pipeline(X_subset)
        else:
            X_preprocessed = self._transform_pipeline(X_subset)

        final_features = keep_available(selected_features, X_preprocessed.columns)
        return X_preprocessed[final_features], y_subset

    def _assert_fully_fitted(self):
        if self._pipeline is None or self._selected_features is None:
            raise RuntimeError(
                "Preprocessor is not fully fitted. Call fit_transform() on training data first."
            )

# Train test split

In [28]:
train_df, test_df = train_test_split(
    nacc_raw,
    test_size=0.2,
    random_state=42,
    stratify=nacc_raw['EVENT_MCI'],
)

In [32]:
import wandb
from scipy.stats import randint, uniform

features_num = X_train_for_tunning.shape[1]
n_samples = X_train_for_tunning.shape[0]

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="semariik",
    # Set the wandb project where this run will be logged.
    project="survival-analysis-mci",
    # Track hyperparameters and run metadata.
    config={
        "model":            "RandomSurvivalForest",
        "tuning_strategy":  "RandomizedSearchCV",
        "n_iter":           10,
        "cv_folds":         5,
        "n_samples":        n_samples,
        "n_features":       features_num,
        "scoring":          "concordance_index",
        "dataset":          "MCI",
        "param_space": {
            "num_trees":       "randint(1, 2001)",
            "mtry":            "['sqrt', 'log2']",
            "min_node_size":   "n^x transformation",
            "replace":         "[True, False]",
            "sample_fraction": "uniform(0.1, 0.9)",
        }
    },
)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mariik04 (semariik) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Final dataset preprocessing

In [ ]:
import joblib
best_parameters = joblib.load("joblib-storage/rsf_vimp_best_params.joblib")

In [ ]:
data_preprocessor = FullDatasetPreprocessor(best_rsf_vimp_parameters=best_parameters)
X_train, y_train = data_preprocessor.fit_transform(df=train_df)
X_test, y_test = data_preprocessor.transform(df=test_df)

In [ ]:
X_train_reset = X_train.reset_index(drop=True)
X_test_reset = X_test.reset_index(drop=True)

In [ ]:
X_train_reset.shape, X_test_reset.shape

In [ ]:
train_df = pd.concat([X_train_reset, pd.DataFrame(y_train)], axis=1)
test_df = pd.concat([X_test_reset, pd.DataFrame(y_test)], axis=1)

train_df.to_csv("train_preprocessed.csv", index=False)
test_df.to_csv("test_preprocessed.csv", index=False)

In [ ]:
plt.figure(figsize=(15, 8))
bars = sns.barplot(x=train_df['EVENT_MCI'].value_counts().index, y=train_df['EVENT_MCI'].value_counts().values)
for bar in bars.containers:
    ax = bars.axes
    ax.bar_label(bar, padding=3, fontsize=8)
plt.title('Distribution of EVENT_MCI')
plt.xlabel('EVENT_MCI')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
train_df['EVENT_MCI'].value_counts(normalize=True)